# Module B — LightGBM Price Forecasting Model
**Project**: TOP (Tomato, Onion, Potato) Digital Twin  
**Target Journal**: Computers and Electronics in Agriculture / Agricultural Systems  
**Author**: Dr. Masroor, SKUAST-K (HADP-04)  

## Pipeline
1. Load master weekly panel (all 6 data sources merged)
2. Feature engineering (lags, rolling stats, seasonality dummies)
3. Train / validation / test split (temporal)
4. LightGBM training with Optuna hyperparameter tuning
5. Evaluation: RMSE, MAE, MAPE, R²
6. Feature importance + SHAP values
7. Export results for paper tables and figures

---
## 0. Imports and Config

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Paths ────────────────────────────────────────────────────────────
BASE        = r'C:\Users\masro\Downloads'
AGM_DIR     = os.path.join(BASE, 'Agmarknet_Weekly')
CMIE_DIR    = os.path.join(BASE, 'CMIE_Macro')
RBI_DIR     = os.path.join(BASE, 'RBI_DBIE')
PPAC_DIR    = os.path.join(BASE, 'PPAC_Macro')
SAT_DIR     = os.path.join(BASE, 'TOP_DT_Satellite', 'S2')
ERA5_DIR    = os.path.join(BASE, 'ERA5')
OUT_DIR     = os.path.join(BASE, 'Model_Output')
os.makedirs(OUT_DIR, exist_ok=True)

CROPS = ['tomato', 'onion', 'potato']

# Study window
TRAIN_END   = '2022-12-31'
VAL_END     = '2023-12-31'
# Test = 2024 (held out)

print('Paths OK')
print(f'Output directory: {OUT_DIR}')

---
## 1. Load Data Sources

In [ ]:
# ── 1a. Agmarknet Weekly Panel ────────────────────────────────────────
agm = pd.read_csv(os.path.join(AGM_DIR, 'top_weekly_panel.csv'), parse_dates=['week_start'])
print(f'Agmarknet: {len(agm):,} rows  |  crops: {agm["crop"].unique()}')
agm.head(3)

In [ ]:
# ── 1b. CMIE Macro (monthly → will be joined on year-month) ──────────
cmie = pd.read_csv(os.path.join(CMIE_DIR, 'cmie_macro_2017_2024.csv'), parse_dates=['date'])
print(f'CMIE macro: {len(cmie)} rows  |  cols: {list(cmie.columns)}')
cmie.head(3)

In [ ]:
# ── 1c. RBI DBIE Macro ───────────────────────────────────────────────
rbi = pd.read_csv(os.path.join(RBI_DIR, 'rbi_dbie_macro_2017_2024.csv'), parse_dates=['date'])
print(f'RBI DBIE: {len(rbi)} rows  |  cols: {list(rbi.columns)}')
rbi.head(3)

In [ ]:
# ── 1d. PPAC Fuel Prices ─────────────────────────────────────────────
ppac = pd.read_csv(os.path.join(PPAC_DIR, 'ppac_diesel_lpg_2017_2024.csv'), parse_dates=['date'])
print(f'PPAC: {len(ppac)} rows  |  cols: {list(ppac.columns)}')
ppac.head(3)

In [ ]:
# ── 1e. Satellite S2 (if available) ──────────────────────────────────
sat_file = os.path.join(SAT_DIR, 's2_weekly_vi.csv')   # adjust filename as needed
if os.path.exists(sat_file):
    sat = pd.read_csv(sat_file, parse_dates=['week_start'])
    print(f'Satellite S2: {len(sat):,} rows  |  cols: {list(sat.columns)}')
else:
    sat = None
    print('Satellite S2 file not found — will run without satellite features.')

In [ ]:
# ── 1f. ERA5 Weather (if available) ──────────────────────────────────
era5_file = os.path.join(ERA5_DIR, 'era5_weekly.csv')   # adjust filename as needed
if os.path.exists(era5_file):
    era5 = pd.read_csv(era5_file, parse_dates=['week_start'])
    print(f'ERA5: {len(era5):,} rows  |  cols: {list(era5.columns)}')
else:
    era5 = None
    print('ERA5 file not found — will run without weather features.')

---
## 2. Build Monthly Macro Table
CMIE, RBI and PPAC are monthly. Merge them into one macro table, then join to weekly panel on year-month.

In [ ]:
macro = (
    cmie
    .merge(rbi,  on='date', how='outer', suffixes=('', '_rbi'))
    .merge(ppac, on='date', how='outer', suffixes=('', '_ppac'))
    .sort_values('date')
    .reset_index(drop=True)
)
# Drop duplicate year/month cols that came from rbi/ppac
macro = macro[[c for c in macro.columns if not c.endswith(('_rbi', '_ppac'))]]
macro['ym'] = macro['date'].dt.to_period('M')

print(f'Macro table: {len(macro)} rows  |  {len(macro.columns)} cols')
print(macro.isnull().sum().to_string())
macro.tail(3)

---
## 3. Feature Engineering

In [ ]:
def build_features(crop_name: str, panel: pd.DataFrame, macro: pd.DataFrame,
                   sat=None, era5=None) -> pd.DataFrame:
    """
    Build feature matrix for one crop.
    Returns a DataFrame with target = modal_price_weighted (t+1 week ahead).
    """
    df = panel[panel['crop'] == crop_name].copy()
    df = df.sort_values(['market_id', 'week_start']).reset_index(drop=True)

    # ── Calendar features ────────────────────────────────────────────
    df['month']      = df['week_start'].dt.month
    df['week_of_yr'] = df['week_start'].dt.isocalendar().week.astype(int)
    df['sin_week']   = np.sin(2 * np.pi * df['week_of_yr'] / 52)
    df['cos_week']   = np.cos(2 * np.pi * df['week_of_yr'] / 52)
    df['sin_month']  = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month']  = np.cos(2 * np.pi * df['month'] / 12)

    # ── Price lags (per market) ───────────────────────────────────────
    for lag in [1, 2, 3, 4, 8, 13, 26, 52]:
        df[f'price_lag_{lag}'] = (
            df.groupby('market_id')['modal_price_weighted']
              .shift(lag)
        )

    # ── Rolling stats ─────────────────────────────────────────────────
    g = df.groupby('market_id')['modal_price_weighted']
    for w in [4, 8, 13]:
        df[f'price_roll_mean_{w}'] = g.transform(lambda x: x.shift(1).rolling(w).mean())
        df[f'price_roll_std_{w}']  = g.transform(lambda x: x.shift(1).rolling(w).std())

    # ── Arrivals features ─────────────────────────────────────────────
    df['arr_lag_1']       = df.groupby('market_id')['arrivals_tonnes_week'].shift(1)
    df['arr_roll_mean_4'] = (
        df.groupby('market_id')['arrivals_tonnes_week']
          .transform(lambda x: x.shift(1).rolling(4).mean())
    )

    # ── Year-over-year price change ───────────────────────────────────
    df['price_yoy'] = (
        df.groupby(['market_id', 'week_of_yr'])['modal_price_weighted']
          .shift(1)  # same week last year (approx)
    )
    df['price_yoy_pct'] = (df['modal_price_weighted'] - df['price_yoy']) / df['price_yoy']

    # ── Join monthly macro on year-month ──────────────────────────────
    df['ym'] = df['week_start'].dt.to_period('M')
    macro_cols = [c for c in macro.columns if c != 'date' and c not in ['year','month']]
    df = df.merge(macro[macro_cols], on='ym', how='left')

    # ── Satellite VI (optional) ───────────────────────────────────────
    if sat is not None:
        sat_crop = sat[sat['crop'] == crop_name] if 'crop' in sat.columns else sat
        df = df.merge(sat_crop.drop(columns=['crop'], errors='ignore'),
                      on='week_start', how='left')

    # ── ERA5 weather (optional) ───────────────────────────────────────
    if era5 is not None:
        era5_crop = era5[era5['crop'] == crop_name] if 'crop' in era5.columns else era5
        df = df.merge(era5_crop.drop(columns=['crop'], errors='ignore'),
                      on='week_start', how='left')

    # ── Target: next-week price ───────────────────────────────────────
    df['target'] = df.groupby('market_id')['modal_price_weighted'].shift(-1)

    # Drop rows with NaN target
    df = df.dropna(subset=['target'])

    return df


print('Feature builder defined.')

In [ ]:
# Build for each crop
feat = {}
for crop in CROPS:
    feat[crop] = build_features(crop, agm, macro, sat=sat, era5=era5)
    print(f'{crop}: {feat[crop].shape}  |  NaN in target: {feat[crop]["target"].isna().sum()}')

feat['tomato'].head(2)

---
## 4. Train / Validation / Test Split
- **Train**: 2017–2022
- **Val**:   2023
- **Test**:  2024 (held out, never used for tuning)

In [ ]:
EXCLUDE_COLS = [
    'crop', 'state', 'state_code', 'district', 'market',
    'week_start', 'ym', 'target',
    'modal_price_weighted',  # current-week price (leaks into target)
]

splits = {}  # crop → {X_train, y_train, X_val, y_val, X_test, y_test, feature_cols}

for crop in CROPS:
    df = feat[crop]

    train = df[df['week_start'] <= TRAIN_END]
    val   = df[(df['week_start'] > TRAIN_END) & (df['week_start'] <= VAL_END)]
    test  = df[df['week_start'] > VAL_END]

    feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
    # Encode state and market_id as integer categories
    for cat_col in ['state_code', 'market_id']:
        if cat_col in df.columns:
            cat = pd.Categorical(df[cat_col])
            df[cat_col + '_enc'] = cat.codes
            if cat_col + '_enc' not in feature_cols:
                feature_cols.append(cat_col + '_enc')

    splits[crop] = {
        'X_train': train[feature_cols],
        'y_train': train['target'],
        'X_val':   val[feature_cols],
        'y_val':   val['target'],
        'X_test':  test[feature_cols],
        'y_test':  test['target'],
        'feature_cols': feature_cols,
        'test_df': test,
    }
    print(f'{crop:8s}  train={len(train):,}  val={len(val):,}  test={len(test):,}  features={len(feature_cols)}')

---
## 5. LightGBM Training (Baseline)

In [ ]:
# Install if needed: pip install lightgbm optuna shap
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


BASE_PARAMS = {
    'objective':        'regression',
    'metric':           'rmse',
    'learning_rate':    0.05,
    'num_leaves':       63,
    'min_child_samples':20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'lambda_l1':        0.1,
    'lambda_l2':        0.1,
    'verbose':         -1,
    'n_jobs':          -1,
    'seed':             42,
}

models   = {}
metrics  = {}

for crop in CROPS:
    s = splits[crop]
    X_tr, y_tr = s['X_train'], s['y_train']
    X_va, y_va = s['X_val'],   s['y_val']
    X_te, y_te = s['X_test'],  s['y_test']

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_va, label=y_va, reference=dtrain)

    model = lgb.train(
        BASE_PARAMS,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200),
        ]
    )
    models[crop] = model

    pred_te = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred_te))
    mae  = mean_absolute_error(y_te, pred_te)
    mpe  = mape(y_te.values, pred_te)
    r2   = r2_score(y_te, pred_te)

    metrics[crop] = {'RMSE': rmse, 'MAE': mae, 'MAPE%': mpe, 'R2': r2,
                     'best_iter': model.best_iteration}
    print(f'{crop:8s}  RMSE={rmse:.1f}  MAE={mae:.1f}  MAPE={mpe:.1f}%  R²={r2:.3f}')

In [ ]:
# Summary table — paper Table 4
metrics_df = pd.DataFrame(metrics).T.round(3)
metrics_df.index.name = 'Crop'
print(metrics_df.to_string())
metrics_df.to_csv(os.path.join(OUT_DIR, 'table4_model_metrics.csv'))

---
## 6. Hyperparameter Tuning (Optuna)
Run once — takes ~20-40 min per crop. Skip to Section 7 if baseline metrics are sufficient.

In [ ]:
# Uncomment to run Optuna tuning

# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# def objective(trial, crop):
#     s = splits[crop]
#     params = {
#         'objective':        'regression',
#         'metric':           'rmse',
#         'verbose':         -1,
#         'n_jobs':          -1,
#         'seed':             42,
#         'learning_rate':    trial.suggest_float('lr', 0.01, 0.1, log=True),
#         'num_leaves':       trial.suggest_int('num_leaves', 31, 255),
#         'min_child_samples':trial.suggest_int('min_child_samples', 10, 100),
#         'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
#         'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
#         'bagging_freq':     trial.suggest_int('bagging_freq', 1, 10),
#         'lambda_l1':        trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
#         'lambda_l2':        trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
#     }
#     dtrain = lgb.Dataset(s['X_train'], label=s['y_train'])
#     dval   = lgb.Dataset(s['X_val'],   label=s['y_val'], reference=dtrain)
#     model  = lgb.train(params, dtrain, num_boost_round=2000,
#                        valid_sets=[dval],
#                        callbacks=[lgb.early_stopping(100, verbose=False),
#                                   lgb.log_evaluation(period=-1)])
#     pred = model.predict(s['X_val'])
#     return np.sqrt(mean_squared_error(s['y_val'], pred))
#
# best_models = {}
# for crop in CROPS:
#     study = optuna.create_study(direction='minimize',
#                                  sampler=optuna.samplers.TPESampler(seed=42))
#     study.optimize(lambda t: objective(t, crop), n_trials=100, show_progress_bar=True)
#     print(f'{crop}: best RMSE={study.best_value:.1f}  params={study.best_params}')

print('Optuna block ready — uncomment to run.')

---
## 7. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, crop in zip(axes, CROPS):
    model = models[crop]
    imp   = pd.Series(model.feature_importance(importance_type='gain'),
                      index=splits[crop]['feature_cols'])
    imp   = imp.sort_values(ascending=False).head(20)
    imp   = imp.sort_values()   # ascending for horizontal bar

    ax.barh(imp.index, imp.values, color='steelblue')
    ax.set_title(f'{crop.capitalize()} — Top 20 Features (Gain)', fontsize=12)
    ax.set_xlabel('Gain')
    ax.tick_params(labelsize=8)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_feature_importance.png'), dpi=200, bbox_inches='tight')
plt.show()
print('Saved: fig_feature_importance.png')

---
## 8. SHAP Values

In [ ]:
import shap

shap_values = {}

for crop in CROPS:
    s = splits[crop]
    explainer = shap.TreeExplainer(models[crop])
    # Use test set (subsample for speed)
    X_sample = s['X_test'].sample(min(2000, len(s['X_test'])), random_state=42)
    sv = explainer.shap_values(X_sample)
    shap_values[crop] = (sv, X_sample)
    print(f'{crop}: SHAP computed on {len(X_sample)} test samples')

In [ ]:
# SHAP beeswarm plots — paper Figure 5
for crop in CROPS:
    sv, X_sample = shap_values[crop]
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, X_sample, max_display=15, show=False,
                      title=f'{crop.capitalize()} — SHAP Summary')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'fig_shap_{crop}.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: fig_shap_{crop}.png')

---
## 9. Actual vs Predicted Plot (Test Set 2024)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, crop in zip(axes, CROPS):
    s  = splits[crop]
    te = s['test_df'].copy()
    te['pred'] = models[crop].predict(s['X_test'])

    # National weekly average
    grp = te.groupby('week_start').agg(
        actual=('target', 'mean'),
        predicted=('pred', 'mean')
    ).reset_index()

    ax.plot(grp['week_start'], grp['actual'],    label='Actual',    linewidth=1.5)
    ax.plot(grp['week_start'], grp['predicted'], label='Predicted', linewidth=1.5, linestyle='--')
    ax.set_title(f'{crop.capitalize()} — Test Set 2024 (National Weekly Mean)', fontsize=11)
    ax.set_ylabel('Price (Rs/quintal)')
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    m = metrics[crop]
    ax.text(0.02, 0.92,
            f'RMSE={m["RMSE"]:.0f}  MAE={m["MAE"]:.0f}  MAPE={m["MAPE%"]:.1f}%  R²={m["R2"]:.3f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_actual_vs_pred_2024.png'), dpi=200, bbox_inches='tight')
plt.show()
print('Saved: fig_actual_vs_pred_2024.png')

---
## 10. Save Final Predictions

In [ ]:
all_preds = []

for crop in CROPS:
    s  = splits[crop]
    te = s['test_df'][['crop','state','market_id','market','week_start','target']].copy()
    te['predicted'] = models[crop].predict(s['X_test'])
    te['residual']  = te['target'] - te['predicted']
    all_preds.append(te)

preds_df = pd.concat(all_preds, ignore_index=True)
preds_df.to_csv(os.path.join(OUT_DIR, 'test_predictions_2024.csv'), index=False)
print(f'Saved: test_predictions_2024.csv  ({len(preds_df):,} rows)')
preds_df.head(5)

In [ ]:
# Save models
for crop in CROPS:
    mpath = os.path.join(OUT_DIR, f'lgbm_{crop}.txt')
    models[crop].save_model(mpath)
    print(f'Model saved: {mpath}')

print('\nAll outputs in:', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:<45} {sz:>7.1f} KB')